# Generate poems
Qwen writes the poem, this notebook controls the form:
- line length from the poet's profile (syllables)
- the last word of each line is picked from 300 candidates, scored `λ · p(w)/p_max + (1 − λ) · rhyme(w, partner)`
- stops after 3 quatrains (rhymed poets) or when the model wants to end (unrhymed poets)

Where the 300 candidates come from:
- B1: Qwen's own 300 most likely next words (B0 = B1 at λ 1)
- M: the meaning head (`meaning_head.ipynb`), the 300 words closest to what Qwen "means"

Every line end is logged in a trace.

In [1]:
%%capture
%run rhyme.ipynb

In [2]:
import json
import math
import os
import random
import numpy as np
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_DIR = "../model/Qwen3-0.6B-Base"
LORA_DIR = "../lora_weights/"  # the trained weights from GitHub (README step 6)
HEAD_FILE = "../lora_weights/meaning_head.pt"  # from meaning_head.ipynb
TRACES = "../outputs/traces/"
try:
    QUIET  # evaluate.ipynb sets this to True, then the examples at the bottom are skipped
except NameError:
    QUIET = False

TEMP = 0.8
TEMP_FIRST_LINE = 1.0  # so poems by the same poet don't all start the same way
TOP_P = 0.95
REP = 1.1  # light repetition penalty
K = 300  # candidate words for a line end
MU = 0.05  # weight of rhyme richness when a line opens a new rhyme

profiles = pd.read_csv(DATA + "author_profiles.csv", index_col=0)
showcase = profiles[profiles["unknown_ends"] <= 0.1].sort_values("lines", ascending=False).groupby("scheme").head(1)
print(showcase[["scheme", "syl_min", "syl_max", "lam"]])

torch.set_grad_enabled(False)  # only generating, otherwise memory fills up with gradients
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to("cuda")
if os.path.exists(LORA_DIR + "adapter_config.json"):
    # merged into the model: faster, and the meaning head was trained on the merged model's hidden states
    model = PeftModel.from_pretrained(model, LORA_DIR).merge_and_unload()
    MODEL_NAME = "lora"
else:
    MODEL_NAME = "base"
model.eval()
print("model:", MODEL_NAME)

W1120 19:55:57.776000 10432 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


                           scheme  syl_min  syl_max  lam
poet                                                    
Henry Wadsworth Longfellow   none        8       10  1.0
Madison Julius Cawein        ABCB        7       10  0.5
Algernon Charles Swinburne   AABB        8       11  0.5
George Gordon Byron          ABAB       10       11  0.5


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 550.54it/s]


model: lora


## Words that can end a line
In the vocabulary from `meaning_head.ipynb` (GloVe + CMUdict), with a vowel, and ended at least 5 lines in the
training poems. That keeps out word starts that happen to be in CMUdict (" clo" from "cloud", " w" from "wind").
The same list for B1 and M, and the same rhyme richness table.

For B1 the word also has to be a single Qwen token, words Qwen splits up can't come out of B1.

In [3]:
line_ends = Counter()
for text in train["text"]:
    for line in poem_lines(text):
        w = words_of(line)
        if w:
            line_ends[w[-1]] += 1

saved_vocab = np.load("../lora_weights/vocab.npz")
vocab_index = {w: i for i, w in enumerate(saved_vocab["words"])}
allowed = [str(w) for w in saved_vocab["words"] if re.search("[aeiouy]", w) and line_ends[w] >= 5]
allowed_idx = {w: i for i, w in enumerate(allowed)}
rich_vocab = {w: int(saved_vocab["richness"][vocab_index[w]]) for w in allowed}

vocab = [tokenizer.decode([i]) for i in range(len(tokenizer))]
tok_ids, tok_word = [], []  # tokens that are a whole line-end word (" light", " Light")
for i, s in enumerate(vocab):
    if re.fullmatch(r" [A-Za-z]+", s) and s[1:].lower() in allowed_idx:
        tok_ids.append(i)
        tok_word.append(allowed_idx[s[1:].lower()])
tok_ids, tok_word = torch.tensor(tok_ids, device="cuda"), torch.tensor(tok_word, device="cuda")

EOS = tokenizer.eos_token_id
mid_line_ban = torch.tensor([i for i, s in enumerate(vocab) if "\n" in s] + [EOS], device="cuda")
line_breaks = torch.tensor([i for i, s in enumerate(vocab) if re.fullmatch(r"[,.;:!?]?\n", s)], device="cuda")

print(len(allowed), "line-end words,", len(tok_ids), "of them as single tokens (B1)")

head = None
if os.path.exists(HEAD_FILE):
    saved = torch.load(HEAD_FILE)
    assert saved["words"] == allowed, "meaning head was trained on a different word list, rerun meaning_head.ipynb"
    head = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1024), nn.GELU(), nn.Linear(1024, 300)).cuda()
    head.load_state_dict(saved["head"])
    head.eval()
    TAU = saved["tau"]
    V = torch.tensor(saved_vocab["vectors"][[vocab_index[w] for w in allowed]], dtype=torch.float32, device="cuda")
    V = V / V.norm(dim=1, keepdim=True)
print("meaning head:", "loaded" if head is not None else "not trained yet, only B1")

18395 line-end words, 16708 of them as single tokens (B1)
meaning head: loaded


## Writing mid-line
Sample the next token (temperature, top-p, repetition penalty). Line breaks and the end of the poem are
blocked here, the line end is decided below. `forward` also returns the hidden state, the meaning head reads it.

In [4]:
def forward(ids, past=None):
    out = model.model(input_ids=torch.tensor([ids], device="cuda"), past_key_values=past, use_cache=True)
    h = out.last_hidden_state[0, -1]
    return model.lm_head(h).float(), out.past_key_values, h


def sample(logits, temp, seen, ban=None, only=None):
    logits = logits.clone()
    if seen:
        s = torch.tensor(list(seen), device="cuda")
        logits[s] = torch.where(logits[s] > 0, logits[s] / REP, logits[s] * REP)
    if ban is not None:
        logits[ban] = -float("inf")
    if only is not None:
        keep = torch.full_like(logits, -float("inf"))
        keep[only] = logits[only]
        logits = keep
    probs = torch.softmax(logits / temp, -1)
    p, order = probs.sort(descending=True)
    p[(p.cumsum(0) - p) > TOP_P] = 0  # top-p
    return order[torch.multinomial(p, 1)].item()

## The last word of a line
p = Qwen's probability of the word for B1, the meaning head's for M (`softmax(cos / τ)`, what it was trained with;
raw cosines are all about the same size so λ would hardly do anything).

Take the 300 most likely, keep the ones that make the line land within 1 syllable of its target, then:
- the line has a rhyme partner: `λ · p/p_max + (1 − λ) · rhyme(w, partner)`
- the line opens a new rhyme: `λ · p/p_max + μ · log(1 + richness(w))`, so it doesn't start with *orange*
- free line (lines 1 and 3 in ABCB, unrhymed poets): just `p/p_max`

In [5]:
def word_probs(system, logits, h):
    if system == "B1":
        p = torch.softmax(logits, -1)[tok_ids]
        return torch.zeros(len(allowed), device="cuda").index_add_(0, tok_word, p)
    v = head(h.float()[None])[0]
    cos = V @ (v / v.norm())
    return torch.softmax(cos / TAU, -1)


def line_end(system, logits, h, line, target, partner, opening, lam):
    p_all = word_probs(system, logits, h)
    top = p_all.topk(K)
    p = {allowed[i]: pr for pr, i in zip(top.values.tolist(), top.indices.tolist())}

    have = line_syllables(line)
    off = {w: abs(have + syllables(w) - target) for w in p if w != partner}
    fits = [w for w in off if off[w] <= max(1, min(off.values()))]
    p_max = max(p[w] for w in fits)

    cands = []
    for w in fits:
        r = rhyme(w, partner) if partner else 0
        if partner:
            score = lam * p[w] / p_max + (1 - lam) * r
        elif opening:
            score = lam * p[w] / p_max + MU * math.log(1 + rich_vocab[w])
        else:
            score = p[w] / p_max
        cands.append({"word": w, "p": round(p[w], 4), "rhyme": r, "score": round(score, 3)})
    cands.sort(key=lambda c: (-c["score"], -c["p"]))
    return cands, list(p)[0]  # (the candidates, the system's own favourite word)


def word_tokens(word, logits):
    # write the word with the spelling Qwen likes most (" God" not " god"), or as several tokens if it has to
    best, best_p = None, -1
    for t in [tokenizer.encode(" " + word), tokenizer.encode(" " + word.capitalize())]:
        if len(t) == 1 and logits[t[0]] > best_p:
            best, best_p = t, logits[t[0]]
    return best or tokenizer.encode(" " + word)

## Which line rhymes with which
Line numbers inside each quatrain (0-3). `partner` = the line it has to rhyme with, `opens` = it starts a new rhyme.

In [6]:
PARTNER = {"AABB": {1: 0, 3: 2}, "ABAB": {2: 0, 3: 1}, "ABCB": {3: 1}, "none": {}}
OPENS = {"AABB": {0, 2}, "ABAB": {0, 1}, "ABCB": {1}, "none": set()}

for scheme in ["AABB", "ABAB", "ABCB"]:
    print(scheme, "partners:", PARTNER[scheme], "opens:", OPENS[scheme])

AABB partners: {1: 0, 3: 2} opens: {0, 2}
ABAB partners: {2: 0, 3: 1} opens: {0, 1}
ABCB partners: {3: 1} opens: {1}


## The whole poem
Rhymed poets stop after 3 quatrains. Unrhymed poets stop when the model's end-of-text probability at the
start of a line is over 0.5 after at least 8 lines, or at 20 lines.

In [7]:
def generate_poem(poet, lam=None, seed=0, system="B1"):
    prof = profiles.loc[poet]
    scheme = prof["scheme"]
    lam = prof["lam"] if lam is None else lam
    random.seed(seed)
    torch.manual_seed(seed)
    logits, past, h = forward(tokenizer(f"Author: {poet}\n\n")["input_ids"])

    lines, ends, trace, seen = [], [], [], set()
    stop = "3 quatrains" if scheme != "none" else "20 lines"
    while len(lines) < (12 if scheme != "none" else 20):
        i = len(lines)
        if scheme == "none" and i >= 8 and torch.softmax(logits, -1)[EOS] > 0.5:
            stop = "end of text"
            break
        target = random.randint(prof["syl_min"], prof["syl_max"])
        q = i % 4
        partner = ends[i - q + PARTNER[scheme][q]] if q in PARTNER[scheme] else None
        temp = TEMP_FIRST_LINE if i == 0 else TEMP

        line, n_tok = "", 0
        while True:
            tok = sample(logits, temp, seen, ban=mid_line_ban)
            near_end = line_syllables(line) >= target - 3 and vocab[tok].startswith(" ")
            # safety nets: line already too long, or lots of tokens without letters (dashes, quotes) so it never gets there
            if near_end or line_syllables(line) > target + 3 or n_tok >= 40:
                break
            n_tok += 1
            line += vocab[tok]
            seen.add(tok)
            logits, past, h = forward([tok], past)

        # the last word, then a line break (with or without , . ; etc)
        qwen_top = allowed[word_probs("B1", logits, h).argmax().item()]
        cands, meant = line_end(system, logits, h, line, target, partner, q in OPENS[scheme], lam)
        word = cands[0]["word"]
        for t in word_tokens(word, logits):
            line += vocab[t]
            seen.add(t)
            logits, past, h = forward([t], past)
        tok = sample(logits, TEMP, set(), only=line_breaks)
        line += vocab[tok].rstrip("\n")
        logits, past, h = forward([tok], past)

        line = line.strip()
        lines.append(line[:1].upper() + line[1:])
        ends.append(word)
        trace.append({"line": i + 1, "text": lines[-1], "syllables": line_syllables(line), "target": target,
                      "partner": partner, "qwen_top": qwen_top, "meant": meant, "chosen": word, "candidates": cands[:10]})

    return {"poet": poet, "system": system, "lam": lam, "scheme": scheme, "seed": seed, "model": MODEL_NAME,
            "stop": stop, "poem": "\n".join(lines), "lines": trace}

## Try it
One poem per showcase poet with B1 and M, at λ 1 (no rhyme pressure), 0.5 and 0 (only rhyme counts).
Longfellow doesn't rhyme, so only λ 1.

In [8]:
runs = []
if not QUIET:
    for system in (["B1", "M"] if head is not None else ["B1"]):
        for poet in showcase.index:
            for lam in ([1.0, 0.5, 0.0] if showcase.loc[poet, "scheme"] != "none" else [1.0]):
                r = generate_poem(poet, lam=lam, seed=0, system=system)
                runs.append(r)
                print(f"===== {system}: {poet} ({r['scheme']}, λ {lam}, stopped: {r['stop']}) =====")
                print(r["poem"])
                print()

===== B1: Henry Wadsworth Longfellow (none, λ 1.0, stopped: 20 lines) =====
On the isle of Sarsunya the
Venus sits in splendor; behold,
In her own narrow chamber
The three daughters sit beside her,
And mingle their voices at even.
Alas, who could have dreamed before
Their flight was planned for this hour!
Thou that dost smile on every face,
Dost greet with all thy friendship each
Close companion to-morrow! Ah,
I know not if thee I can forget;
For once thou lovedst Venus!
Since thou hast known her smiles and faces,
Ere to my heart's good-natured eyes
A shadow fell a moment ago,
Let me remember, remembering,
What she said when we parted.
So saying it I sat him waiting,
And though he called me 'thee' again
By the same name long since in heaven,
===== B1: Madison Julius Cawein (ABCB, λ 1.0, stopped: 3 quatrains) =====
How the falling leaves fall on our
Curved fence - how they are fallen away
In small clusters, scattered
Above and below it everywhere
From field to field: - so much as our
Li

## Checks
Does every line fit the poet's length, and do the lines that should rhyme rhyme (score ≥ 0.9)?
Plus one line end from an M trace: what the head meant, what Qwen wanted, and what got picked.

In [9]:
for r in runs:
    prof = profiles.loc[r["poet"]]
    syl = [t["syllables"] for t in r["lines"]]
    in_range = np.mean([prof["syl_min"] - 1 <= s <= prof["syl_max"] + 1 for s in syl])
    pairs = [(t["chosen"], t["partner"]) for t in r["lines"] if t["partner"]]
    rhymed = np.mean([rhyme(a, b) >= 0.9 for a, b in pairs]) if pairs else float("nan")
    print(f"{r['system']:<3} {r['poet']:<28} λ {r['lam']}: {len(syl)} lines, {in_range:.0%} right length, {rhymed:.0%} of {len(pairs)} rhyme pairs rhyme")

m_runs = [r for r in runs if r["system"] == "M" and r["lam"] == 0.5]
if m_runs:
    t = [t for t in m_runs[0]["lines"] if t["partner"]][0]
    print(f"\n{m_runs[0]['poet']}, line {t['line']}: has to rhyme with '{t['partner']}', the head meant '{t['meant']}', "
          f"Qwen wanted '{t['qwen_top']}', chose '{t['chosen']}'")
    print(pd.DataFrame(t["candidates"][:5]).to_string(index=False))

B1  Henry Wadsworth Longfellow   λ 1.0: 20 lines, 100% right length, nan% of 0 rhyme pairs rhyme
B1  Madison Julius Cawein        λ 1.0: 12 lines, 100% right length, 0% of 3 rhyme pairs rhyme
B1  Madison Julius Cawein        λ 0.5: 12 lines, 100% right length, 33% of 3 rhyme pairs rhyme
B1  Madison Julius Cawein        λ 0.0: 12 lines, 100% right length, 100% of 3 rhyme pairs rhyme
B1  Algernon Charles Swinburne   λ 1.0: 12 lines, 100% right length, 0% of 6 rhyme pairs rhyme
B1  Algernon Charles Swinburne   λ 0.5: 12 lines, 100% right length, 50% of 6 rhyme pairs rhyme
B1  Algernon Charles Swinburne   λ 0.0: 12 lines, 100% right length, 50% of 6 rhyme pairs rhyme
B1  George Gordon Byron          λ 1.0: 12 lines, 100% right length, 0% of 6 rhyme pairs rhyme
B1  George Gordon Byron          λ 0.5: 12 lines, 100% right length, 33% of 6 rhyme pairs rhyme
B1  George Gordon Byron          λ 0.0: 12 lines, 100% right length, 83% of 6 rhyme pairs rhyme
M   Henry Wadsworth Longfellow   λ 1.0: 2

In [10]:
if runs:
    os.makedirs(TRACES, exist_ok=True)
    for r in runs:
        name = f"{r['system']}_{r['poet'].replace(' ', '_')}_lam{r['lam']}_seed{r['seed']}.json"
        with open(TRACES + name, "w", encoding="utf-8") as f:
            json.dump(r, f, indent=1, ensure_ascii=False)
    print("saved", len(runs), "traces in", TRACES)

saved 20 traces in ../outputs/traces/
